# Unit Test: Oracle Enterprise Agents

**Copyright 2026, Denis Rothman**

This notebook validates the functionality of the Enterprise Agents module created in Chapter 4. It replicates the environment setup of Chapters 2 & 3 but uses the new modular architecture (`oracle_lib`, `agent_archivist`, `agent_recruiter`).

### Prerequisities
Ensure you have uploaded the following files to the content root (`/content/`):
1. `oracle_lib.py`
2. `agent_archivist.py`
3. `agent_recruiter.py`
4. `helpers.py`

# 1.Install Dependencies

In [ ]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


In [ ]:
# 1. Install Dependencies
!pip install oracledb==3.4.1 openai==2.14.0 tiktoken==0.7.0 tqdm==4.67.1 --quiet
print("✅ Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.7 MB/s eta 0:00:00
✅ Dependencies installed.


# 2.Setup Environment

In [ ]:
# 2. Setup Environment
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Load API Key
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise ValueError("API_KEY not found in Secrets.")
    os.environ["OPENAI_API_KEY"] = api_key
    client = OpenAI()
    print("✅ OpenAI Client initialized.")
except Exception as e:
    print(f"❌ Setup Failed: {e}")

Mounted at /content/drive
✅ OpenAI Client initialized.


# 3.Import Enterprise Modules

In [ ]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/Chapter04/unit_library.zip" -o unit_library.zip

time.sleep(10)
!unzip -o unit_library.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8036  100  8036    0     0  29107      0 --:--:-- --:--:-- --:--:-- 29221
Archive:  unit_library.zip
  inflating: /content/agent_recruiter.py  
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/agent_archivist.py  


In [ ]:
# 3. Import Enterprise Modules
# Ensure you have uploaded: oracle_lib.py, agent_archivist.py, agent_recruiter.py, helpers.py
try:
    from oracle_lib import OracleManager
    from agent_archivist import agent_oracle_archivist
    from agent_recruiter import agent_oracle_recruiter
    from helpers import create_mcp_message
    print("✅ Enterprise Modules imported.")
except ImportError as e:
    print(f"❌ Import Failed: {e}")
    print("Please ensure oracle_lib.py, agent_archivist.py, agent_recruiter.py, and helpers.py are uploaded to /content/")

✅ Enterprise Modules imported.


 4.Initialize Database Connection

In [ ]:
# 4. Initialize Database Connection
try:
    OracleManager.initialize()
    print("✅ Oracle Connection Manager initialized.")
except Exception as e:
    print(f"❌ Oracle Init Failed: {e}")

✅ Oracle Connection Manager initialized.


# 5.Test Case A: The Archivist (Unstructured)

In [ ]:
# 5. Test Case A: The Archivist (Unstructured)
print("--- TEST CASE A: Oracle Archivist ---")

archivist_input = create_mcp_message("Tester", {
    "intent_query": "What happened during the cyber incident?"
})

try:
    response_a = agent_oracle_archivist(archivist_input, client)
    content_a = response_a.get("content", {})

    if "retrieved_context" in content_a:
        print(f"\n[Result] Source: {content_a.get('source')}")
        print(f"[Result] Context Snippet: {content_a['retrieved_context'][:100]}...")
        print("✅ Archivist Test Passed.")
    else:
        print(f"❌ Archivist returned unexpected format: {response_a}")

except Exception as e:
    print(f"❌ Archivist Test Failed: {e}")

--- TEST CASE A: Oracle Archivist ---

[Result] Source: Oracle 23ai (KNOWLEDGE_STORE)
[Result] Context Snippet: Software Release Notes: Hotfix v2.4.1 Release Date: 2025-10-24 Status: DEPLOYED TO STAGING  Summary:...
✅ Archivist Test Passed.


# 6.Test Case B: The Recruiter (Hybrid)

In [ ]:
# 6. Test Case B: The Recruiter (Hybrid)
print("--- TEST CASE B: Oracle Recruiter ---")

recruiter_input = create_mcp_message("Tester", {
    "intent_query": "Find a Python Team Lead",
    "constraints": {
        "max_salary": 200000,
        "min_experience": 4
    }
})

try:
    response_b = agent_oracle_recruiter(recruiter_input, client)
    content_b = response_b.get("content", {})
    candidates = content_b.get("candidate_list", [])

    if candidates:
        print(f"\n[Result] Source: {content_b.get('source')}")
        print(f"[Result] Candidates Found: {len(candidates)}")
        print(f"[Result] Top Candidate: {candidates[0]['name']} (Score: {candidates[0]['match_score']:.3f})")
        print("✅ Recruiter Test Passed.")
    else:
        print("⚠️ Recruiter found no candidates (Check DB data), but execution was successful.")

except Exception as e:
    print(f"❌ Recruiter Test Failed: {e}")

--- TEST CASE B: Oracle Recruiter ---

[Result] Source: Oracle 23ai (Hybrid Search)
[Result] Candidates Found: 3
[Result] Top Candidate: Riley S. (Score: -0.226)
✅ Recruiter Test Passed.


In [ ]:
# 7. Cleanup
OracleManager.close()
print("✅ Connection closed.")

✅ Connection closed.
